# Aalborg road network – getting started

This notebook shows how to load the data, look at it on a map, turn it into a graph
and draw a tour. See `README.md` for what every column means.

Needs: `pandas`, `shapely` (for the map) and `pyarrow` (only for the `.parquet` files).

In [ ]:
import pandas as pd
import roadmap                      # roadmap.py in this folder

edges = pd.read_csv("edges.csv")    # or: pd.read_parquet("edges.parquet")
nodes = pd.read_csv("nodes.csv")
print(f"{len(nodes):,} nodes, {len(edges):,} arcs")
edges.head()

## How much road is there per priority?

A two-way street is two arcs, so arc-km is about twice the street length.

In [ ]:
summary = edges.groupby("priority").agg(arcs=("edge_id", "size"), km=("length_m", "sum"))
summary["km"] = (summary["km"] / 1000).round(1)
summary

## Look at one salting route on the map

Hover a road to see its attributes; use *Colour by* to switch between priority, speed limit, …

In [ ]:
route = edges[edges["route"] == 301]
print(f"Route 301: {len(route)} arcs, {route['length_m'].sum() / 1000:.1f} km")
roadmap.show_map(route, title="Salting route 301")

## A graph in plain Python

`out_arcs[node]` lists the arcs leaving a node. Together with `head[arc]` (where an
arc ends) and `length[arc]` that is all you need to walk around the network.

In [ ]:
out_arcs = [[] for _ in range(len(nodes))]
for edge_id, u in zip(edges["edge_id"], edges["u"]):
    out_arcs[u].append(edge_id)

tail   = edges["u"].tolist()          # tail[arc] = node the arc starts at
head   = edges["v"].tolist()          # head[arc] = node the arc ends at
length = edges["length_m"].tolist()

start = tail[route["edge_id"].iloc[0]]
print(f"Arcs leaving node {start}:")
for a in out_arcs[start]:
    print(f"  arc {a} -> node {head[a]}, {length[a]} m, priority {edges.at[a, 'priority']}")

## Checking a tour

A tour is a list of `edge_id`s in driving order. It is valid when every arc starts
where the previous one ended. A *closed* tour also ends where it started.

In [ ]:
def check_tour(tour, required):
    """Print whether a tour is connected and closed, and how much of `required` it covers."""
    jumps = [i for i in range(1, len(tour)) if tail[tour[i]] != head[tour[i - 1]]]
    closed = len(tour) > 0 and head[tour[-1]] == tail[tour[0]]
    covered = set(tour) & set(required)
    km = sum(length[a] for a in tour) / 1000
    print(f"{len(tour)} arcs, {km:.2f} km | jumps: {len(jumps)} | closed: {closed} | "
          f"required arcs covered: {len(covered)} of {len(required)}")
    return not jumps and closed

## Drawing a tour

This is **not** a good tour – it is a random walk, just to show how drawing works.
Building a good one is your job. The walk prefers arcs of route 301 it has not driven yet.

In [ ]:
import random
random.seed(1)

required = route["edge_id"].tolist()
todo = set(required)
node, walk = start, []
for _ in range(80):
    options = out_arcs[node]
    fresh = [a for a in options if a in todo]
    a = random.choice(fresh) if fresh else random.choice(options)
    walk.append(a)
    todo.discard(a)
    node = head[a]

check_tour(walk, required)
# draw route 301 as background; network=edges lets the walk leave the route
roadmap.show_map(route, tour=walk, network=edges, tour_label="Random walk (80 arcs)")

## Straight-line distance between nodes

`x_m` / `y_m` are in meters, so the straight-line distance is plain Pythagoras.
The real driving distance is always at least this long – useful for heuristics.

In [ ]:
import math
x, y = nodes["x_m"].tolist(), nodes["y_m"].tolist()

def straight_line_m(a, b):
    return math.hypot(x[a] - x[b], y[a] - y[b])

print(f"{straight_line_m(start, head[walk[-1]]):.0f} m between the start and the end of the walk")